# 37 — RML-Guided Pruning

## Can residues and triggers guide pruning?

Notebook 31 introduced revision triggers.

Notebook 37 combines:

- residues
- budgets
- triggers

into a simple pruning decision framework.

Outputs:

```text
results/37_rml_guided_pruning.csv
figures/37_rml_guided_pruning.png
```

In [ ]:
from pathlib import Path
import os, sys, subprocess

REPO_URL = "https://github.com/thinkthoughts/pruning-rml.git"
REPO_NAME = "pruning-rml"

cwd = Path.cwd().resolve()

if cwd.name == REPO_NAME:
    repo = cwd
elif Path("/content/pruning-rml").exists():
    repo = Path("/content/pruning-rml")
elif (cwd / REPO_NAME).exists():
    repo = cwd / REPO_NAME
else:
    target = Path("/content") if Path("/content").exists() else cwd
    os.chdir(target)
    subprocess.run(["git","clone",REPO_URL], check=True)
    repo = target / REPO_NAME

os.chdir(repo)

src = repo / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

print(repo)

In [ ]:
pkg = repo / "src" / "pruning_rml"
pkg.mkdir(parents=True, exist_ok=True)

(pkg/"paths.py").write_text('''from pathlib import Path
def ensure_dirs(root, names=("results","figures","reports")):
    root = Path(root)
    out={}
    for n in names:
        p=root/n
        p.mkdir(parents=True, exist_ok=True)
        out[n]=p
    return out
''', encoding="utf-8")

(pkg/"guided.py").write_text('''def guidance_score(residue, trigger, budget):
    return residue + budget - trigger

def recommendation(score):
    if score >= 5:
        return "fine_revision"
    if score >= 2:
        return "structured_revision"
    return "major_revision_or_reset"
''', encoding="utf-8")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from pruning_rml.paths import ensure_dirs

outputs = ensure_dirs(repo)
results = outputs["results"]
figures = outputs["figures"]

In [ ]:
rows = [
    ["high residue, low trigger",4,1,2],
    ["moderate residue",3,2,2],
    ["high trigger pressure",2,4,1],
    ["low residue",1,3,1],
    ["critical state",0,5,0],
]

df = pd.DataFrame(
    rows,
    columns=["condition","residue_strength","trigger_pressure","budget_headroom"]
)

df["guidance_score"] = (
    df["residue_strength"]
    + df["budget_headroom"]
    - df["trigger_pressure"]
)

df

In [ ]:
csv_path = results / "37_rml_guided_pruning.csv"
df.to_csv(csv_path, index=False)
print(csv_path)

In [ ]:
fig_path = figures / "37_rml_guided_pruning.png"

ax = df.plot.bar(
    x="condition",
    y="guidance_score",
    legend=False,
    figsize=(10,5),
)

ax.set_title("RML-guided pruning")
ax.set_ylabel("Guidance score")

plt.tight_layout()
plt.savefig(fig_path, dpi=160)
plt.show()

print(fig_path)

## Result

Notebook 37 combines:

```text
residue
+ budget
- trigger
```

to guide revision decisions.

Next:

```text
41_pruning_policies.ipynb
```